In [1]:
import plotly.graph_objs as go
from plotly.io import show
from sklearn import set_config
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline

from skfolio import Population, RatioMeasure
from skfolio.datasets import load_ftse100_dataset
from skfolio.metrics import make_scorer
from skfolio.model_selection import (
    WalkForward,
    cross_val_predict,
)
from skfolio.optimization import MeanRisk
from skfolio.pre_selection import SelectKExtremes
from skfolio.preprocessing import prices_to_returns

prices = load_ftse100_dataset()
X = prices_to_returns(prices)
X_train, X_test = train_test_split(X, test_size=0.33, shuffle=False)

In [2]:
benchmark = MeanRisk()

In [3]:
set_config(transform_output="pandas")

model = Pipeline([("pre_selection", SelectKExtremes()), ("optimization", benchmark)])

In [4]:
cv = WalkForward(train_size=252, test_size=60)

scorer = make_scorer(RatioMeasure.ANNUALIZED_SHARPE_RATIO)

In [5]:
grid_search = GridSearchCV(
    estimator=model,
    cv=cv,
    n_jobs=-1,
    param_grid={"pre_selection__k": list(range(5, 66, 3))},
    scoring=scorer,
    return_train_score=True,
)
grid_search.fit(X_train)
model = grid_search.best_estimator_
print(model)

Pipeline(steps=[('pre_selection', SelectKExtremes(k=53)),
                ('optimization', MeanRisk())])


In [6]:
cv_results = grid_search.cv_results_
fig = go.Figure(
    [
        go.Scatter(
            x=cv_results["param_pre_selection__k"],
            y=cv_results["mean_train_score"],
            name="Train",
            mode="lines",
            line=dict(color="rgb(31, 119, 180)"),
        ),
        go.Scatter(
            x=cv_results["param_pre_selection__k"],
            y=cv_results["mean_train_score"] + cv_results["std_train_score"],
            mode="lines",
            line=dict(width=0),
            showlegend=False,
        ),
        go.Scatter(
            x=cv_results["param_pre_selection__k"],
            y=cv_results["mean_train_score"] - cv_results["std_train_score"],
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            fillcolor="rgba(31, 119, 180,0.15)",
            fill="tonexty",
        ),
        go.Scatter(
            x=cv_results["param_pre_selection__k"],
            y=cv_results["mean_test_score"],
            name="Test",
            mode="lines",
            line=dict(color="rgb(255,165,0)"),
        ),
        go.Scatter(
            x=cv_results["param_pre_selection__k"],
            y=cv_results["mean_test_score"] + cv_results["std_test_score"],
            mode="lines",
            line=dict(width=0),
            showlegend=False,
        ),
        go.Scatter(
            x=cv_results["param_pre_selection__k"],
            y=cv_results["mean_test_score"] - cv_results["std_test_score"],
            line=dict(width=0),
            mode="lines",
            fillcolor="rgba(255,165,0, 0.15)",
            fill="tonexty",
            showlegend=False,
        ),
    ]
)
fig.add_vline(
    x=grid_search.best_params_["pre_selection__k"],
    line_width=2,
    line_dash="dash",
    line_color="green",
)
fig.update_layout(
    title="Train/Test score",
    xaxis_title="Number of pre-selected best performers",
    yaxis_title="Annualized Sharpe Ratio",
)
fig.update_yaxes(tickformat=".2f")
show(fig)

In [7]:
pred_bench = cross_val_predict(
    benchmark,
    X_test,
    cv=cv,
    portfolio_params=dict(name="Benchmark"),
)

pred_model = cross_val_predict(
    model,
    X_test,
    cv=cv,
    n_jobs=-1,
    portfolio_params=dict(name="Pre-selection"),
)

In [8]:
population = Population([pred_bench, pred_model])

In [9]:
population.plot_cumulative_returns()

In [10]:
population.plot_composition(display_sub_ptf_name=False)

In [11]:
population.summary()

,Benchmark,Pre-selection
Mean,0.029%,0.032%
Annualized Mean,7.28%,8.16%
Variance,0.0074%,0.0075%
Annualized Variance,1.85%,1.88%
Semi-Variance,0.0040%,0.0041%
Annualized Semi-Variance,1.02%,1.03%
Standard Deviation,0.86%,0.86%
Annualized Standard Deviation,13.61%,13.71%
Semi-Deviation,0.64%,0.64%
Annualized Semi-Deviation,10.09%,10.15%
